# The Fourier Transform: From Signal Decomposition to Forecasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/fourier_transform.ipynb)

Decompose signals with the FFT, filter noise with Gaussian windows, visualise time-frequency structure with the Gabor transform, and forecast by training one neural network per frequency band.

**Blog post:** [The Fourier Transform: From Signal Decomposition to Forecasting](https://sesen.ai/blog/fourier-transform-signal-decomposition-forecasting)

**Key references:**
- Cooley, J.W. & Tukey, J.W. (1965). An algorithm for the machine calculation of complex Fourier series. *Math. Comp.*, 19(90), 297-301.
- Gabor, D. (1946). Theory of communication. *Journal of the IEE*, 93(26), 429-457.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Composite Signal and FFT

In [ ]:
n = 512
t = np.linspace(0, 1, n, endpoint=False)
fs = n  # 512 Hz

f1, f2, f3 = 5, 23, 50
signal = 2*np.sin(2*np.pi*f1*t) + 1.5*np.sin(2*np.pi*f2*t) + 0.8*np.sin(2*np.pi*f3*t)

# FFT
F = np.fft.fft(signal)
freqs = np.fft.fftfreq(n, d=1/fs)
magnitude = 2 * np.abs(F[:n//2]) / n
magnitude[0] /= 2
freq_axis = freqs[:n//2]

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(t, signal, color='#2196F3', linewidth=1)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Composite Signal: {f1} Hz + {f2} Hz + {f3} Hz')
axes[0].grid(True, alpha=0.3)

axes[1].stem(freq_axis, magnitude, linefmt='#FF9800', markerfmt='o', basefmt='gray')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Magnitude')
axes[1].set_title('Frequency Spectrum (FFT)'); axes[1].set_xlim(0, 70)
axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

## 2. Band-Pass Filtering

In [ ]:
def bandpass(F, freqs, f_low, f_high):
    mask = (np.abs(freqs) >= f_low) & (np.abs(freqs) < f_high)
    F_band = np.zeros_like(F)
    F_band[mask] = F[mask]
    return np.real(np.fft.ifft(F_band))

bands = [(0, 15), (15, 35), (35, 70)]
band_names = ['Low (0–15 Hz)', 'Mid (15–35 Hz)', 'High (35–70 Hz)']
sub_signals = [bandpass(F, freqs, lo, hi) for lo, hi in bands]

# Verify linearity
error = np.max(np.abs(signal - sum(sub_signals)))
print(f"Reconstruction error: {error:.2e}")

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
axes[0].plot(t, signal, color='#333333', linewidth=1)
axes[0].set_ylabel('Original'); axes[0].set_title('Band-Pass Decomposition')
axes[0].grid(True, alpha=0.3)
colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, (sub, name, c) in enumerate(zip(sub_signals, band_names, colors)):
    axes[i+1].plot(t, sub, color=c, linewidth=1.2)
    axes[i+1].set_ylabel(name); axes[i+1].grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

## 3. Gaussian Filter for Denoising

In [ ]:
L = 30
n_gauss = 512
t_g = np.linspace(-L, L, n_gauss, endpoint=False)
k = (2*np.pi / (2*L)) * np.concatenate([np.arange(0, n_gauss//2), np.arange(-n_gauss//2, 0)])

clean = 1.0 / np.cosh(t_g)
F_clean = np.fft.fft(clean)

# Add white noise in frequency domain (matching R code)
noise_scale = 10
F_noisy = F_clean + noise_scale * (np.random.randn(n_gauss) + 1j * np.random.randn(n_gauss))
noisy = np.fft.ifft(F_noisy)

# Gaussian filter
def gaussian_filter(F, k, k0, bandwidth):
    return F * np.exp(-bandwidth * (k - k0)**2)

F_filtered = gaussian_filter(F_noisy, k, k0=0, bandwidth=0.2)
filtered = np.fft.ifft(F_filtered)

fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].plot(t_g, clean, color='#2196F3', linewidth=2)
axes[0].set_title('Original: sech(t)'); axes[0].grid(True, alpha=0.3)
axes[1].plot(t_g, np.abs(noisy), color='#E53935', linewidth=0.8)
axes[1].set_title('Noisy (white noise, scale=10)'); axes[1].grid(True, alpha=0.3)
axes[2].plot(t_g, np.abs(filtered), color='#4CAF50', linewidth=2, label='Filtered')
axes[2].plot(t_g, clean, color='#2196F3', linestyle='--', alpha=0.7, label='True')
axes[2].set_title('Recovered (Gaussian filter, k₀=0)'); axes[2].legend()
axes[2].grid(True, alpha=0.3); axes[2].set_xlabel('Time')
fig.tight_layout(); plt.show()

## 4. Signal Averaging

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, reps in zip(axes.flat, [1, 5, 10, 100]):
    avg = np.zeros(n_gauss, dtype=complex)
    for _ in range(reps):
        F_r = F_clean + noise_scale * (np.random.randn(n_gauss) + 1j * np.random.randn(n_gauss))
        avg += F_r
    avg /= reps
    recovered = np.real(np.fft.ifft(avg))
    ax.plot(t_g, recovered, color='#FF9800', linewidth=1.2)
    ax.plot(t_g, clean, color='#2196F3', linestyle='--', alpha=0.5)
    ax.set_title(f'{reps} repetition{"s" if reps > 1 else ""}'); ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.5, 1.3)
fig.suptitle('Signal Averaging', fontsize=13, y=1.02)
fig.tight_layout(); plt.show()

## 5. Gabor Transform (Spectrogram)

In [ ]:
L_g = 10
n_g = 2048
t_gabor = np.linspace(0, L_g, n_g, endpoint=False)
k_g = (2*np.pi / L_g) * np.concatenate([np.arange(0, n_g//2), np.arange(-n_g//2, 0)])

# Complex non-stationary signal (faithful to R code)
S = (3*np.sin(2*t_gabor) + 0.5*np.tanh(0.5*(t_gabor-3)) +
     0.2*np.exp(-(t_gabor-4)**2) + 1.5*np.sin(5*t_gabor) +
     4*np.cos(3*(t_gabor-6)**2)) / 10 + (t_gabor/20)**3

# Compute spectrogram
window_centers = np.linspace(0, L_g, 200)
bandwidth = 1.0
freq_gabor = k_g[:n_g//2] / (2*np.pi)
freq_mask = freq_gabor < 8

spectrogram = np.zeros((n_g//2, len(window_centers)))
for j, tau in enumerate(window_centers):
    window = np.exp(-bandwidth * (t_gabor - tau)**2)
    F_w = np.fft.fft(S * window)
    spectrogram[:, j] = np.abs(F_w[:n_g//2])

fig, axes = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 2]})
axes[0].plot(t_gabor, S, color='#2196F3', linewidth=1)
axes[0].set_ylabel('Amplitude'); axes[0].set_title('Non-Stationary Signal')
axes[0].grid(True, alpha=0.3); axes[0].set_xlim(0, L_g)
im = axes[1].pcolormesh(window_centers, freq_gabor[freq_mask],
                         spectrogram[freq_mask, :], cmap='YlOrRd', shading='auto')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title('Gabor Spectrogram')
plt.colorbar(im, ax=axes[1], label='Magnitude', shrink=0.8)
fig.tight_layout(); plt.show()

## 6. Forecasting with FFT + Neural Networks

In [ ]:
from sklearn.neural_network import MLPRegressor

# Signal with known frequency components
n_total = 400
t_fc = np.linspace(0, 10, n_total)
y = (2*np.sin(2*np.pi*0.5*t_fc) + np.sin(2*np.pi*2.0*t_fc) +
     0.5*np.sin(2*np.pi*5.0*t_fc) + 0.15*t_fc)

n_train = 320
train_y = y[:n_train]
test_y = y[n_train:]

# FFT and band-pass decomposition
F_fc = np.fft.fft(train_y)
freqs_fc = np.fft.fftfreq(n_train, d=t_fc[1]-t_fc[0])

def bandpass_hz(F, freqs, f_low, f_high):
    mask = (np.abs(freqs) >= f_low) & (np.abs(freqs) < f_high)
    F_band = np.zeros_like(F)
    F_band[mask] = F[mask]
    return np.real(np.fft.ifft(F_band))

fs_fc = 1.0 / (t_fc[1]-t_fc[0])
bands_hz = [(0, 0.3), (0.3, 1.2), (1.2, 3.5), (3.5, fs_fc/2)]
band_names_fc = ['Trend', '0.5 Hz', '2.0 Hz', '5.0 Hz']
subs = [bandpass_hz(F_fc, freqs_fc, lo, hi) for lo, hi in bands_hz]

print(f"Bands: {len(subs)}, Reconstruction error: {np.max(np.abs(train_y - sum(subs))):.2e}")

In [ ]:
twindow = 4

def make_windows(data, w):
    X = np.column_stack([data[i:len(data)-w+i] for i in range(w-1, -1, -1)])
    y_out = data[w:]
    return X[:len(y_out)], y_out

# Train MLPs
models, fitted_vals = [], []
for i, sub in enumerate(subs):
    X_tr, y_tr = make_windows(sub, twindow)
    mlp = MLPRegressor(hidden_layer_sizes=(4,), max_iter=3000,
                       random_state=42, activation='tanh', solver='lbfgs', tol=1e-10)
    mlp.fit(X_tr, y_tr)
    models.append(mlp)
    fitted_vals.append(mlp.predict(X_tr))
    print(f"  {band_names_fc[i]:>8s}: R²={mlp.score(X_tr, y_tr):.4f}")

# Forecast auto-regressively
n_forecast = len(test_y)
forecasts = []
for model, sub in zip(models, subs):
    X_tr, _ = make_windows(sub, twindow)
    current = X_tr[-1:].copy()
    preds = []
    for _ in range(n_forecast):
        pred = model.predict(current)[0]
        preds.append(pred)
        current = np.roll(current, 1)
        current[0, 0] = pred
    forecasts.append(np.array(preds))

insample = sum(fitted_vals)
outsample = sum(forecasts)

In [ ]:
# Plot forecast
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.arange(len(y)), y, color='#333333', linewidth=1.5, label='True signal', alpha=0.8)
ax.plot(np.arange(twindow, n_train), insample, color='#2196F3', linewidth=1.5,
        linestyle='--', label='In-sample fit', alpha=0.8)
ax.plot(np.arange(n_train, n_train+n_forecast), outsample, color='#E53935',
        linewidth=2, label='Forecast')
ax.axvline(n_train, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Time Step'); ax.set_ylabel('Amplitude')
ax.set_title('FFT Band-Pass + Neural Network Forecasting')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

## Exercises

1. **Add noise to the 3-component signal.** Add Gaussian noise and see how the FFT peaks broaden. At what noise level do the peaks become undetectable?

2. **Build a spectrogram for audio.** Load a `.wav` file and compute its Gabor spectrogram. Can you identify the vowels in a spoken word from their frequency signatures?

3. **Try a financial time series.** Apply band-pass decomposition to SPY daily returns. Do any frequency bands have predictive power?

4. **Compare window functions.** Replace the Gaussian window in the Gabor transform with a rectangular window and a Hann window. How does the spectrogram change?